<a href="https://colab.research.google.com/github/Arav-18/Walmart-Sales-Dashboard/blob/main/YOLOv5_KBS_Imp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
os.environ["WANDB_DISABLED"] = "true"

YOLOv5-KBS

In [5]:
from google.colab import files
uploaded = files.upload()

Saving NEU-DET.zip to NEU-DET (1).zip


In [6]:
import zipfile, os

zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

os.listdir("/content/dataset")


['NEU-DET']

In [7]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
!pip install -r requirements.txt


Cloning into 'yolov5'...
remote: Enumerating objects: 17881, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 17881 (delta 38), reused 8 (delta 8), pack-reused 17828 (from 3)
Receiving objects: 100% (17881/17881), 16.99 MiB | 27.01 MiB/s, done.
Resolving deltas: 100% (12181/12181), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [8]:
import os, random, shutil
from PIL import Image
import xml.etree.ElementTree as ET

base = "/content/dataset/NEU-DET"
CLASSES = ["crazing","inclusion","patches","pitted_surface","rolled-in_scale","scratches"]

def convert(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall("object"):
        cls = obj.find("name").text
        # Ensure class exists in CLASSES, if not, handle it (e.g., skip or warn)
        if cls not in CLASSES:
            print(f"Warning: Class '{cls}' found in XML but not in predefined CLASSES. Skipping object in {xml_path}.")
            continue
        cls_id = CLASSES.index(cls)
        b = obj.find("bndbox")
        xmin, ymin = int(b.find("xmin").text), int(b.find("ymin").text)
        xmax, ymax = int(b.find("xmax").text), int(b.find("ymax").text)
        cx = ((xmin + xmax) / 2) / img_w
        cy = ((ymin + ymax) / 2) / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f"{cls_id} {cx} {cy} {w} {h}")
    return lines

# Define base directories for images and annotations
img_source_dir = os.path.join(base, "IMAGES")
ann_source_dir = os.path.join(base, "ANNOTATIONS")
labels_base_dir = os.path.join(base, "labels")

# Create class-specific label directories
for cls_name in CLASSES:
    os.makedirs(os.path.join(labels_base_dir, cls_name), exist_ok=True)

# Process each image file
for f in os.listdir(img_source_dir):
    if f.endswith(".jpg"):
        img_path = os.path.join(img_source_dir, f)
        xml_path = os.path.join(ann_source_dir, f.replace(".jpg", ".xml"))

        try:
            img = Image.open(img_path)
            yolo_labels = convert(xml_path, *img.size)

            # Determine the class from the image filename (e.g., "crazing_1.jpg" -> "crazing")
            # This assumes a naming convention like "classname_number.jpg"
            # Find the longest matching class prefix in the filename
            cls_name_from_file = None
            for class_name in CLASSES:
                if f.startswith(f"{class_name}_"):
                    cls_name_from_file = class_name
                    break # Found the correct class name

            if cls_name_from_file is not None:
                label_output_dir = os.path.join(labels_base_dir, cls_name_from_file)
                output_label_path = os.path.join(label_output_dir, f.replace(".jpg", ".txt"))
                with open(output_label_path, "w") as fp:
                    fp.write("\n".join(yolo_labels))
            else:
                print(f"Warning: Image filename '{f}' does not match a known class prefix. Labels for this image might not be saved to a specific class folder.")

        except FileNotFoundError as e:
            print(f"Error processing {f}: {e}. Skipping this file.")
        except Exception as e:
            print(f"An unexpected error occurred while processing {f}: {e}. Skipping this file.")

In [9]:
for s in ["train","val"]:
    os.makedirs(f"/content/yolo_dataset/images/{s}", exist_ok=True)
    os.makedirs(f"/content/yolo_dataset/labels/{s}", exist_ok=True)

all_imgs = []
for f in os.listdir(img_source_dir):
    if f.endswith(".jpg"):
        cls_name_from_file = None
        for class_name in CLASSES:
            if f.startswith(f"{class_name}_"):
                cls_name_from_file = class_name
                break

        if cls_name_from_file is not None:
            all_imgs.append((cls_name_from_file, f))
        else:
            print(f"Warning: Image filename '{f}' does not match a known class prefix. Skipping for split.")

random.shuffle(all_imgs)
split_point = int(0.8 * len(all_imgs))

def move(data, split_type):
    for cls, f in data:
        src_img_path = os.path.join(img_source_dir, f)
        dst_img_path = os.path.join(f"/content/yolo_dataset/images/{split_type}", f)
        shutil.copy(src_img_path, dst_img_path)

        src_label_path = os.path.join(labels_base_dir, cls, f.replace(".jpg", ".txt"))
        dst_label_path = os.path.join(f"/content/yolo_dataset/labels/{split_type}", f.replace(".jpg", ".txt"))
        shutil.copy(src_label_path, dst_label_path)

move(all_imgs[:split_point], "train")
move(all_imgs[split_point:], "val")

In [10]:
%%writefile /content/yolo_dataset/data.yaml
train: /content/yolo_dataset/images/train
val: /content/yolo_dataset/images/val

nc: 6
names: ["crazing","inclusion","patches","pitted_surface","rolled-in_scale","scratches"]


Writing /content/yolo_dataset/data.yaml


In [11]:
data_yaml = f"""
path: /content/yolo_dataset  # dataset root dir
train: images/train  # train images (relative to 'path')
val: images/val  # val images (relative to 'path')

nc: {len(CLASSES)}  # number of classes
names: {CLASSES}  # class names

"""

with open("/content/yolov5/data/NEU-DET.yaml", 'w') as f:
    f.write(data_yaml)

print("NEU-DET.yaml created successfully!")

NEU-DET.yaml created successfully!


In [12]:
from pathlib import Path

common_path = Path("/content/yolov5/models/common.py")
text = common_path.read_text()

if "SELayer" not in text:
    text += """

# ==========================
# SE Attention (YOLOv5-KBS)
# ==========================
class SELayer(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(c, c // r, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(c // r, c, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y
"""

# Inject SE into existing C3
text = text.replace(
    "class C3(nn.Module):",
    "class C3(nn.Module):\n    use_se = True"
)

text = text.replace(
    "def forward(self, x):\n        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))",
    """def forward(self, x):
        y = self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))
        if getattr(self, 'use_se', False):
            y = SELayer(y.shape[1])(y)
        return y"""
)

common_path.write_text(text)
print(" SE successfully injected into C3")


 SE successfully injected into C3


In [13]:
!cp models/yolov5m.yaml models/yolov5m-kbs.yaml

!python train.p3y \
--img 832 \
--batch 16 \
--epochs 150 \
--data /content/yolo_dataset/data.yaml \
--cfg models/yolov5m-kbs.yaml \
--weights yolov5m.pt \
--name yolov5_kbs_0799 \
--patience 50 \
--cache \
--exist-ok

Streaming output truncated to the last 5000 lines.
    123/149      12.1G    0.02279    0.02313  0.0004728         68        832:   6% 5/90 [00:03<01:02,  1.37it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    123/149      12.1G    0.02358    0.02316  0.0005884         75        832:   7% 6/90 [00:04<01:01,  1.35it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    123/149      12.1G    0.02402    0.02302   0.000563         67        832:   8% 7/90 [00:05<01:02,  1.34it/s]/content/yolov5/train.py:414: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(amp):
    123/149      12.1G    0.02415     0.

In [17]:
!zip -r yolov5_KBS_FINAL_RESULTS.zip runs/train/yolov5_kbs_0799

  adding: runs/train/yolov5_kbs_0799/ (stored 0%)
  adding: runs/train/yolov5_kbs_0799/val_batch1_pred.jpg (deflated 8%)
  adding: runs/train/yolov5_kbs_0799/val_batch0_labels.jpg (deflated 17%)
  adding: runs/train/yolov5_kbs_0799/labels_correlogram.jpg (deflated 37%)
  adding: runs/train/yolov5_kbs_0799/results.csv (deflated 83%)
  adding: runs/train/yolov5_kbs_0799/PR_curve.png (deflated 9%)
  adding: runs/train/yolov5_kbs_0799/confusion_matrix.png (deflated 22%)
  adding: runs/train/yolov5_kbs_0799/R_curve.png (deflated 6%)
  adding: runs/train/yolov5_kbs_0799/train_batch2.jpg (deflated 6%)
  adding: runs/train/yolov5_kbs_0799/events.out.tfevents.1776074134.5c81ecca83ed.11985.0 (deflated 29%)
  adding: runs/train/yolov5_kbs_0799/val_batch2_pred.jpg (deflated 15%)
  adding: runs/train/yolov5_kbs_0799/weights/ (stored 0%)
  adding: runs/train/yolov5_kbs_0799/weights/last.pt (deflated 8%)
  adding: runs/train/yolov5_kbs_0799/weights/best.pt (deflated 8%)
  adding: runs/train/yolov5_kb

In [18]:
from google.colab import files
files.download("yolov5_KBS_FINAL_RESULTS.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
!cp yolov5_KBS_FINAL_RESULTS.zip /content/drive/MyDrive/

In [ ]:
!ls /content/drive/MyDrive | grep yolov5


yolov5_KBS_FINAL_RESULTS.zip


In [ ]:
!cp runs/train/yolov5_kbs/weights/best.pt /content/drive/MyDrive/yolov5_kbs_best.pt


In [31]:
import time
import torch
import os
import glob
import cv2
from models.common import DetectMultiBackend
from utils.torch_utils import select_device
from utils.general import check_img_size

# -----------------------------
# LOAD MODEL
# -----------------------------
weights = "runs/train/yolov5_kbs_0799/weights/best.pt"

device = select_device('')
model = DetectMultiBackend(weights, device=device)

imgsz = 832
imgsz = check_img_size(imgsz)

# -----------------------------
# AUTO LOAD IMAGES
# -----------------------------
img_folder = "/content/yolo_dataset/images/val" # Changed from '/content/yolo_dataset/images/test'
image_paths = glob.glob(os.path.join(img_folder, "*.jpg"))

images = []
for path in image_paths[:20]:   # use 20 images
    img = cv2.imread(path)
    img = cv2.resize(img, (imgsz, imgsz))
    img = img[:, :, ::-1].transpose(2, 0, 1).copy() # Added .copy() here
    img = torch.from_numpy(img).float() / 255.0
    img = img.unsqueeze(0).to(device)
    images.append(img)

# -----------------------------
# WARMUP
# -----------------------------
for _ in range(10):
    if images: # Check if images list is not empty before accessing images[0]
        model(images[0])
    else:
        print("Warning: No images found in the specified folder for warm-up.")
        break # Exit warm-up if no images

# -----------------------------
# FPS MEASUREMENT
# -----------------------------
start = time.time()

for img in images:
    model(img)

end = time.time()

if len(images) > 0:
    fps = len(images) / (end - start)
    print(f" FPS: {fps:.2f*2}")
    print(f" Time per image: {1000/fps:.2f} ms")
else:
    print("No images processed for FPS measurement.")

YOLOv5 🚀 v7.0-467-g1d62daa3 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
YOLOv5m-kbs summary: 212 layers, 20873139 parameters, 0 gradients, 47.9 GFLOPs


 FPS: 78.01
 Time per image: 25.64 ms
